# IOAI — 2025 Stage 3 Translation Stylization (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, subprocess, zipfile, urllib.request
subprocess.run(['pip','install','-q','sentencepiece','sacremoses'])
if not os.path.exists('data/train_dataset.jsonl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-translation-stylization/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 번역 문체화 모범답안 (미세조정)

폴란드 AI 올림피아드 II · 2025 · 결선. EN→PL 번역모델을 미세조정해 **전문용어를 영어로 유지**하는 문체 학습.

**방법**: `gsarti/opus-mt-tc-en-pl` 를 train(2808쌍, en→문체화 pl)에 몇 에폭 미세조정. 토크나이저는 그대로.
`process_example` 은 항등(en 입력). 모델이 데이터에서 "용어는 영어로 두는" 스타일을 학습한다.

**성능(valid 856, 실측)**: BLEU **0.866** → **100/100** (베이스라인 0.73 → 0점).

**제출**: `submission.json` — 생성 번역 리스트.


In [ ]:
# 데이터 준비 + 모델 로드
import os, json, urllib.request, zipfile
if not os.path.exists("data/train_dataset.jsonl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-3-translation-stylization/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
dev = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "gsarti/opus-mt-tc-en-pl"

def load_jsonl(p):
    out = []
    for line in open(p, encoding="utf-8"):
        it = json.loads(line)
        out.append({"en": it["translation"]["en"], "pl": it["translation"]["pl"],
                    "keywords": ",".join(it.get("keywords", []))})
    return out
train = load_jsonl("data/train_dataset.jsonl"); valid = load_jsonl("data/valid_dataset.jsonl")

tokenizer = AutoTokenizer.from_pretrained(MODEL)   # 토크나이저 변경 금지
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).float().to(dev)   # fp32 (일부 버전은 fp16 로 로드→학습 NaN)
model.lm_head.weight = model.model.shared.weight   # 출력투영을 임베딩과 tie (일부 transformers 버전 로딩 보정)
print("train", len(train), "valid", len(valid), "| dev", dev)

def process_example(en: str, keywords: str) -> str:
    return en                                       # 항등 입력 (미세조정이 문체를 담당)

def generate_all(m, examples, bs=64):
    m.eval(); hyps = []
    for i in range(0, len(examples), bs):
        b = examples[i:i+bs]
        inp = [process_example(e["en"], e["keywords"]) for e in b]
        enc = tokenizer(inp, return_tensors="pt", padding=True, truncation=True, max_length=512).to(dev)
        with torch.no_grad(): out = m.generate(**enc, max_new_tokens=64, num_beams=4)
        hyps += tokenizer.batch_decode(out, skip_special_tokens=True)
    return hyps


In [ ]:
# 모범답안: train 쌍(en -> 문체화 pl)으로 미세조정
import random
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
model.train()
for ep in range(3):
    random.shuffle(train); tot = 0; n = 0
    for i in range(0, len(train), 16):
        b = train[i:i+16]
        enc = tokenizer([e["en"] for e in b], return_tensors="pt", padding=True, truncation=True, max_length=512).to(dev)
        lab = tokenizer(text_target=[e["pl"] for e in b], return_tensors="pt", padding=True, truncation=True, max_length=512).input_ids.to(dev)
        lab[lab == tokenizer.pad_token_id] = -100
        loss = model(**enc, labels=lab).loss
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); n += 1
    print(f"epoch {ep+1}/3 loss {tot/n:.4f}", flush=True)
my_model = model


In [ ]:
# valid 생성 -> submission.json
hyps = generate_all(my_model, valid)
assert len(hyps) == len(valid)
json.dump(hyps, open("submission.json", "w"), ensure_ascii=False)
print("submission.json 저장:", len(hyps), "문장")
print("예시:", hyps[0][:90])


### 정리
- 사전학습 EN→PL 모델을 문체화 데이터로 3에폭 미세조정 → 전문용어를 영어로 유지 → BLEU 0.87 → 100점.
- **핵심**: 데이터가 곧 스타일. 토크나이저는 그대로 두고 모델만 미세조정. (문자단위 BLEU 라 임계값이 0.82~0.86 로 높음.)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)